In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# 경로 설정
target_dir = "/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Basic-Info/de"

# 환자별 노이즈 정보
noise_df = pd.read_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
noise_df['new_pt_id'] = noise_df['new_pt_id'].astype(str)

pt_id_map = noise_df.set_index('pt_id')['new_pt_id'].to_dict()
date_noise_map = noise_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_noise_map = noise_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(target_dir, '**/*.csv'), recursive = True)
print(f"총 처리 대상 파일 수 : {len(csv_files)}")



In [ ]:
# ① pt_id 치환
print("\n① pt_id 치환 시작")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')
        
        if 'pt_id' in df.columns:
            df['pt_id'] = df['pt_id'].map(pt_id_map).fillna(df['pt_id'])
            
        print(df.head(3))
        
        df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"✅ pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ pt_id 오류] {file} → {e}")

In [ ]:
# ② hos_dt ±7일 ±30분
print("\n② hos_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'hos_dt' in df.columns:
            df['hos_dt'] = pd.to_datetime(df['hos_dt'], errors='coerce')
            df['hos_dt'] = df['hos_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ hos_dt 오류] {file} → {e}")



In [ ]:
# ③ birth_date → day=1 고정
print("\n③ birth_date 1일 고정")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'birth_date' in df.columns:
            df['birth_date'] = pd.to_datetime(df['birth_date'], errors='coerce')
            df['birth_date'] = df['birth_date'].apply(lambda x: x.replace(day=1) if pd.notnull(x) else pd.NaT)
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ birth_date 1일 고정 완료: {file}")
    except Exception as e:
        print(f"[❗ birth_date 오류] {file} → {e}")



In [ ]:
# ④ death_date ±7일
print("\n④ death_date 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'death_date' in df.columns:
            df['death_date'] = pd.to_datetime(df['death_date'], errors='coerce')
            df['death_date'] = df['death_date'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0)))
            
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ death_date 완료: {file}")
    except Exception as e:
        print(f"[❗ death_date 오류] {file} → {e}")





In [ ]:
# ⑤ death_est_date ±7일
print("\n⑤ death_est_date 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'death_est_date' in df.columns:
            df['death_est_date'] = pd.to_datetime(df['death_est_date'], errors='coerce')
            df['death_est_date'] = df['death_est_date'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ death_est_date 완료: {file}")
    except Exception as e:
        print(f"[❗ death_est_date 오류] {file} → {e}")



In [ ]:
# ⑥ Region_cd 범주화
print("\n⑥ Region_cd 범주화")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'Region_cd' in df.columns:
            df['Region_cd'] = df['Region_cd'].apply(
                lambda x: x if pd.isna(x) or x == 'KR' else 'Others')
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ Region_cd 범주화 완료: {file}")
    except Exception as e:
        print(f"[❗ Region_cd 오류] {file} → {e}")

In [ ]:
# ⑦ Region_nm 범주화
print("\n⑦ Region_nm 범주화")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'Region_nm' in df.columns:
            df['Region_nm'] = df['Region_nm'].apply(
                lambda x: x if pd.isna(x) or x == '대한민국' else '국외')
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ Region_nm 범주화 완료: {file}")
    except Exception as e:
        print(f"[❗ Region_nm 오류] {file} → {e}")


In [ ]:
# ⑧ 민감 정보 컬럼 삭제
print("\n⑧ 특이/민감 컬럼 삭제")
drop_cols = ['abroad', 'zip_code', 'privacy', 'sensitive', 'unique']
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
        
        print(df.head(3))
            
        df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"✅ 민감 컬럼 삭제 완료: {file}")
    except Exception as e:
        print(f"[❗ 컬럼 삭제 오류] {file} → {e}")


In [ ]:
#  특이 정보 추출 (death_cause_cd, death_caluse_nm N수 5 이하)

rare_combinations_all = []
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if {'death_cause_cd', 'death_cause_nm'}.issubset(df.columns):
            combo_counts = df.groupby(['death_cause_cd', 'death_cause_nm']).size().reset_index(name = 'count')
            rare_combos = combo_counts[combo_counts['count'] <= 5].copy()
            rare_combos['source_file'] = os.path.basename(file)
            rare_combinations_all.append(rare_combos)
            print(f"처리 완료: {file}")
        else:
            print(f"해당 컬럼 없음: {file}")
            
    except Exception as e:
        print(f"오류 발생: {file} -> {e}")
        
if rare_combinations_all:
    result_df = pd.concat(rare_combinations_all, ignore_index = True)
    result_df = result_df.sort_values(by='count', ascending = True).reset_index(drop = True)
    print(result_df.head())
else:
    print("조건에 맞는 조합 없음")
print(f"총 count 수: {result_df['count'].sum()}")

In [ ]:
# 특이 정보 제거(사망원인 N수 5이하 제거)

import os
import glob
import pandas as pd

# 희귀 조합 (result_df 기준)
rare_combos = result_df[['death_cause_cd', 'death_cause_nm']].dropna().drop_duplicates()

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id': str}, sep=',', quotechar='"', lineterminator='\n')

        if {'death_cause_cd', 'death_cause_nm'}.issubset(df.columns):
            before_rows = len(df)
            
            merged = df.merge(rare_combos, on = ['death_cause_cd', 'death_cause_nm'], how = 'left', indicator = True)
            df_filtered = merged[merged['_merge'] == 'left_only'].drop(columns = ['_merge'])
            
            after_rows = len(df_filtered)
            removed_count = before_rows - after_rows
            
            df_filtered.to_csv(file,index = False, encoding = 'utf-8')
            
            print(f"{file}: {removed_count}개 행 제거됨 (남은 행: {after_rows})")
            
        else:
            print(f"❗ 해당 컬럼 없음: {file}")

    except Exception as e:
        print(f"❌ 오류 발생: {file} → {e}")
